In [1]:
import sys
import os
import keras_tuner
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense, Layer, Dropout
from tensorflow.keras import layers
from tensorflow.keras.layers import PReLU


# Get the absolute path of the current script's directory
current_dir = os.path.dirname(os.path.abspath("transformer0.ipynb"))

# Get the absolute path of the parent directory (project_folder)
parent_dir = os.path.dirname(current_dir)

# Add the parent directory to the Python path
sys.path.append(parent_dir)

from FNN1_1 import baseline_deviation, baeline_out_deviation, baseline_long_deviation, baseline_relError, absSum
baseline_out_deviation = baeline_out_deviation

from GetXY_new import x_train, y_train, x_val, y_val, x_test, y_test, out_x_test, out_y_test, long_x_test, long_y_test, outsideExpr, absSum

C:\Users\A_118784\Desktop\matura_github\myenv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\A_118784\Desktop\matura_github\myenv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\A_118784\Desktop\matura_github\myenv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. P

Epoch 1/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 20.1268 - val_loss: 15.9742
Epoch 2/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 16.9567 - val_loss: 14.8742
Epoch 3/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.6945 - val_loss: 13.7262
Epoch 4/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 14.1002 - val_loss: 12.0868
Epoch 5/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 11.9334 - val_loss: 9.7534
Epoch 6/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 9.2567 - val_loss: 7.1517
Epoch 7/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 6.5133 - val_loss: 4.7812
Epoch 8/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.2322 - val_loss: 3.1201
Epoch 9/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.5777 - val_loss: 1.9397
Epoch 10/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.5069 - val_loss: 1.2383
Epoch 11/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9780 - val_loss: 0.9157
Epoch 12/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - 

C:\Users\A_118784\Desktop\matura_github\matura\matura\FNN1_1.py:246: RuntimeWarning: divide by zero encountered in divide
  relativeError = np.where(np.array(y_test) != 0, deviation.flatten() / np.abs(np.array(y_test)), deviation.flatten())


-3.5 + 2.0 + -4.0
15483
-5.5

Expressions not in x:
1 + -4 - -1
5324
-2.0
15
0.0
[ 5.   -0.75 -5.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.  ]


In [2]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=10,
    min_delta=0.0001,
    restore_best_weights=True,
    monitor='val_loss',
    mode = "min"
)

In [3]:
batch_size = 32
print(f"training dataset length: {len(x_train)}")
print(f"validation dataset length: {len(x_val)}")
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(buffer_size=2048).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(batch_size)

training dataset length: 14708
validation dataset length: 775


In [4]:
input_shape = x_train[0].shape
from tensorflow.keras import backend as K

def build_model(hp, input_shape):
    K.clear_session()
    num_neurons = hp.Int("num_neurons", 1, 512)
    num_layers = hp.Int("num_layers", 1, 16)
    dropoutTF = hp.Boolean("dropoutT/F")
    lil_model = keras.Sequential()
    lil_model.add(keras.Input(shape=input_shape))
    lil_model.add(layers.Flatten())
    for i in range(num_layers):
        lil_model.add(layers.Dense(num_neurons)),
        lil_model.add(PReLU())
        if dropoutTF == True:
            lil_model.add(layers.Dropout(0.1))
    lil_model.add(layers.Dense(1, activation='linear'))
    lil_model.compile(optimizer="adam", loss="mse")
    
    early_stopping = keras.callbacks.EarlyStopping(
        patience=20,
        min_delta=0,
        restore_best_weights=True,
        monitor='val_loss',
        mode = "min"
    )
    
    return lil_model
build_model(keras_tuner.HyperParameters(), input_shape)

<Sequential name=sequential, built=True>

In [5]:
import gc
from tensorflow.keras import backend as K

class ClearMemory(keras.callbacks.Callback):
    def on_train_end(self, logs=None):
        # Clear the Keras session to free up the graph
        K.clear_session()
        # Force Python garbage collection
        gc.collect()

In [6]:
tuner = keras_tuner.BayesianOptimization(
    hypermodel=lambda hp: build_model(hp, input_shape),
    objective="val_loss",
    max_trials=50,
    executions_per_trial=1,
    overwrite=False,
    directory="FNNTuner_new",
    project_name="tuner",
)


Reloading Tuner from FNNTuner_new\tuner\tuner0.json


In [7]:
num_epochs = 100
tuner.search(train_dataset, epochs = num_epochs, validation_data = (val_dataset), verbose = 1, callbacks = [ClearMemory(), early_stopping])

In [8]:
best_hps = tuner.get_best_hyperparameters()[0]
best_model = build_model(best_hps, input_shape)
best_model.summary()
tuner.results_summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten (Flatten)                    │ (None, 15)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 6)                   │              96 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ p_re_lu (PReLU)                      │ (None, 6)                   │               6 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │               7 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 109 (436.00 B)

 Trainable params: 109 (436.00 B)

 Non-trainable params: 0 (0.00 B)

Results summary
Results in FNNTuner_new\tuner
Showing 10 best trials
Objective(name="val_loss", direction="min")

Trial 15 summary
Hyperparameters:
num_neurons: 6
num_layers: 1
dropoutT/F: False
Score: 6.15782823842892e-07

Trial 34 summary
Hyperparameters:
num_neurons: 190
num_layers: 1
dropoutT/F: False
Score: 0.00044457532931119204

Trial 29 summary
Hyperparameters:
num_neurons: 110
num_layers: 1
dropoutT/F: False
Score: 0.0005650991224683821

Trial 19 summary
Hyperparameters:
num_neurons: 288
num_layers: 9
dropoutT/F: False
Score: 0.0013342053862288594

Trial 12 summary
Hyperparameters:
num_neurons: 265
num_layers: 1
dropoutT/F: False
Score: 0.001340534770861268

Trial 10 summary
Hyperparameters:
num_neurons: 512
num_layers: 1
dropoutT/F: False
Score: 0.001392784295603633

Trial 33 summary
Hyperparameters:
num_neurons: 206
num_layers: 4
dropoutT/F: False
Score: 0.0019441248150542378

Trial 41 summary
Hyperparameters:
num_neurons: 202
num_layers: 3
dropoutT/F: False
Score: 0.0020359

In [9]:
x_test_dataset = tf.data.Dataset.from_tensor_slices(x_test).batch(batch_size)
out_x_test_dataset = tf.data.Dataset.from_tensor_slices(out_x_test).batch(batch_size)
long_x_test_dataset = tf.data.Dataset.from_tensor_slices(long_x_test).batch(batch_size)

In [10]:
best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=200,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 19.4567 - val_loss: 17.0351
Epoch 2/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 17.9495 - val_loss: 16.5046
Epoch 3/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 17.3222 - val_loss: 15.6483
Epoch 4/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 16.1768 - val_loss: 14.1025
Epoch 5/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 14.2792 - val_loss: 11.8676
Epoch 6/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 12.0398 - val_loss: 9.6641
Epoch 7/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 10.0815 - val_loss: 7.9606
Epoch 8/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8.2865 - val_loss: 6.3152
Epoch 9/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 6.2180 - val_loss: 4.5179
Epoch 10/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 4.2239 - val_loss: 2.9950
Epoch 11/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 2.7909 - val_loss: 1.9927
Epoch 12/200
460/460 ━━━━━━━━

In [11]:
predsInRange = best_model.predict(x_test_dataset)
predsOutRange = best_model.predict(out_x_test_dataset)
predsLongRange = best_model.predict(long_x_test_dataset)

167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


In [12]:
import numpy as np

reldiffInRange = []
diffInRange = []
safe_y_test = np.where(np.isclose(y_test,0.0), 1.0, y_test)

for i in range(len(y_test)):
    diffInRange.append(abs(y_test[i] - predsInRange[i]))
    reldiffInRange.append(abs(y_test[i] - predsInRange[i])/abs(safe_y_test[i]))
print(len(diffInRange))
print("MAE in Range: ", np.mean(diffInRange))
print("MRE in Range: ", np.mean(reldiffInRange))

diffLongRange = []
for i in range(200, 300):
    diffLongRange.append(np.array(np.abs(long_y_test[i]) - np.array(predsLongRange[i])))
    
NEEDdiffLongRange = []
for i in range(len(long_y_test)):
    NEEDdiffLongRange.append(np.array(np.abs(long_y_test[i]) - np.array(predsLongRange[i])))
print("MAE longer Expressions: ", np.mean(NEEDdiffLongRange))

diffOutRange = []
for i in range(len(out_y_test)):
    diffOutRange.append(abs(out_y_test[i] - predsOutRange[i]))
safe_out_y_test = np.where(out_y_test == 0, 1, out_y_test)
diff_out_relError = []
for i in range(len(out_y_test)):
    diff_out_relError.append(abs(diffOutRange[i] / safe_out_y_test[i]))
print("MAE out Range: ", np.mean(diffOutRange))
print("MRE out Range: ", np.mean(diff_out_relError))

5324
MAE in Range:  0.0057218187
MRE in Range:  0.002333898954538289
MAE longer Expressions:  8.007654245240348
MAE out Range:  1.4189942
MRE out Range:  0.17772650261289002


In [13]:
placeholder = absSum(outsideExpr)
diffOutRange = []
indices_with_placeholder_22 = [i for i, val in enumerate(placeholder) if val == 22] 

for i in indices_with_placeholder_22:
    diffOutRange.append(np.abs(out_y_test[i]-predsOutRange[i]))


In [14]:
meanDiff_InRange = np.mean(diffInRange)
meanDiff_OutRange = np.mean(diffOutRange)
meanDiff_LongRange = np.mean(diffLongRange)
meanDiff_OutRelRange = np.mean(diff_out_relError)



In [15]:
benchmark = 0
benchmark += baseline_deviation / (meanDiff_InRange**2) / 4
print(baseline_deviation / (meanDiff_InRange**2) / 4)

benchmark += baseline_out_deviation / (meanDiff_OutRange**2) / 4
print(baseline_out_deviation / (meanDiff_OutRange**2) / 4)

benchmark += baseline_long_deviation / (meanDiff_LongRange**2) / 4
print(baseline_long_deviation / (meanDiff_LongRange**2) / 4)

benchmark += baseline_relError / (meanDiff_OutRelRange**2) / 4
print(baseline_relError / (meanDiff_OutRelRange**2) / 4)

print(f"Benchmark: {benchmark}")

131.3676755844725
0.8256978118387478
0.3296629510660713
0.34388298293242386
Benchmark: 132.86691933030974


In [16]:
# New cell for RNN2 notebook - Statistical analysis with multiple runs
#this is also generated based on FNN6.py (DeepseekR1 this time.) (ofc, slightly tweaked as always)
# Initialize lists to store metrics across multiple runs
benchmarks_rnn = []
MAEinRange_rnn = []
MREinRange_rnn = []
MAEoutRange_rnn = []
MREoutRange_rnn = []
MAElongRange_rnn = []

# Run multiple training iterations
for progress in range(5):
    early_stopping = keras.callbacks.EarlyStopping(
        patience=20,
        min_delta=0,
        restore_best_weights=True,
        monitor='val_loss',
        mode = "min"
    )
    print(f"Progress: {progress + 1}/5")
    
    # Build and train model
    current_model = build_model(best_hps, input_shape)
    
    current_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=200,
        callbacks=[early_stopping],
        verbose=1
    )
    
    # Make predictions
    predsInRange = current_model.predict(x_test_dataset, verbose=0)
    predsOutRange = current_model.predict(out_x_test_dataset, verbose=0)
    predsLongRange = current_model.predict(long_x_test_dataset, verbose=0)
    
    # Calculate metrics
    reldiffInRange = []
    diffInRange = []
    safe_y_test = np.where(np.isclose(y_test, 0.0), 1.0, y_test)
    
    for i in range(len(y_test)):
        diffInRange.append(abs(y_test[i] - predsInRange[i]))
        reldiffInRange.append(abs(y_test[i] - predsInRange[i])/abs(safe_y_test[i]))
    
    MAEinRange_rnn.append(np.mean(diffInRange))
    MREinRange_rnn.append(np.mean(reldiffInRange))

    # Long range metrics
    diffLongRange = []
    for i in range(200, 300):
        diffLongRange.append(np.array(np.abs(long_y_test[i]) - np.array(predsLongRange[i])))
    
    NEEDdiffLongRange = []
    for i in range(len(long_y_test)):
        NEEDdiffLongRange.append(np.array(np.abs(long_y_test[i]) - np.array(predsLongRange[i])))
    MAElongRange_rnn.append(np.mean(NEEDdiffLongRange))
    
    # Out of range metrics
    diffOutRange = []
    for i in range(len(out_y_test)):
        diffOutRange.append(abs(out_y_test[i] - predsOutRange[i]))
    safe_out_y_test = np.where(out_y_test == 0, 1, out_y_test)
    diff_out_relError = []
    for i in range(len(out_y_test)):
        diff_out_relError.append(abs(diffOutRange[i] / safe_out_y_test[i]))
    
    MAEoutRange_rnn.append(np.mean(diffOutRange))
    MREoutRange_rnn.append(np.mean(diff_out_relError))
    
    # Calculate benchmark
    placeholder = absSum(outsideExpr)
    diffOutRange_22 = []
    indices_with_placeholder_22 = [i for i, val in enumerate(placeholder) if val == 22] 
    
    for i in indices_with_placeholder_22:
        diffOutRange_22.append(np.abs(out_y_test[i]-predsOutRange[i]))
    
    meanDiff_InRange = np.mean(diffInRange)
    meanDiff_OutRange = np.mean(diffOutRange_22)
    meanDiff_LongRange = np.mean(diffLongRange)
    meanDiff_OutRelRange = np.mean(diff_out_relError)
    
    benchmark = 0
    benchmark += baseline_deviation / (meanDiff_InRange**2) / 4
    benchmark += baseline_out_deviation / (meanDiff_OutRange**2) / 4
    benchmark += baseline_long_deviation / (meanDiff_LongRange**2) / 4
    benchmark += baseline_relError / (meanDiff_OutRelRange**2) / 4
    
    benchmarks_rnn.append(benchmark)

# Statistical analysis
from scipy.stats import ttest_1samp


stats6, p_value6 = ttest_1samp(benchmarks_rnn, popmean=1)

print(f"FNN Model - Statistical Analysis Results:")

print(f"Benchmark P-value: {p_value6:.6f}")

print(f"\nFNN Model - Average Metrics:")
print(f"Average MAE in Range: {np.mean(MAEinRange_rnn):.6f}")
print(f"Average MRE in Range: {np.mean(MREinRange_rnn):.6f}")
print(f"Average MAE out Range: {np.mean(MAEoutRange_rnn):.6f}")
print(f"Average MRE out Range: {np.mean(MREoutRange_rnn):.6f}")
print(f"Average MAE long Range: {np.mean(MAElongRange_rnn):.6f}")
print(f"Average benchmark: {np.mean(benchmarks_rnn):.6f}")

print(f"\nFNN Model - All Runs:")
print(f"MAE in Range: {[f'{x:.6f}' for x in MAEinRange_rnn]}")
print(f"MRE in Range: {[f'{x:.6f}' for x in MREinRange_rnn]}")
print(f"MAE out Range: {[f'{x:.6f}' for x in MAEoutRange_rnn]}")
print(f"MRE out Range: {[f'{x:.6f}' for x in MREoutRange_rnn]}")
print(f"MAE long Range: {[f'{x:.6f}' for x in MAElongRange_rnn]}")
print(f"Benchmark: {[f'{x:.6f}' for x in benchmarks_rnn]}")

Progress: 1/5
Epoch 1/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 21.7895 - val_loss: 16.8198
Epoch 2/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 18.0491 - val_loss: 16.3468
Epoch 3/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 17.2899 - val_loss: 15.3291
Epoch 4/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 15.7291 - val_loss: 13.3032
Epoch 5/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 12.9997 - val_loss: 10.0786
Epoch 6/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 9.3756 - val_loss: 6.6526
Epoch 7/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 6.0152 - val_loss: 4.0496
Epoch 8/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.6506 - val_loss: 2.4793
Epoch 9/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 2.2603 - val_loss: 1.5601
Epoch 10/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.4450 - val_loss: 1.0111
Epoch 11/200
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9147 - val_loss: 0.6212
Epoch 12/200
460/

In [ ]:
### This actually shows that the benchmark isn't all that great for measuring the performance of the model.
### It shouldn't value an extremely lower Error. There should probably be a limit to how many benchmark-points the model can obtain from each category.
### 
### New proposed benchmark: 
###
### We also saw that the padding = 0 actually didn't have all that big an impact. 
### But I should test for this additionally, by evaluating with the old training data, but with paddding values now set to 0.
###
###
### This notebook has shown, that an increased (6.5x times larger) training dataset, doesn't lead to a better generalization, it only improves the performance on In Range data.